# 06 · 진짜 파이프라인 한 바퀴 돌려보기 (스모크 테스트)

> **공부 기록 노트북 6번.** 01~05 에서 *원리*를 배웠습니다. 이제
> `src/` 의 **진짜 학습 코드**를 돌려봅니다. 단, 끝까지(수 시간) 돌리지 않고
> **epochs=1 + 소수 배치(limit_batches)** 로 "연결만" 확인합니다.

## 왜 이걸 따로 하나

`run_pipeline.ipynb` 는 분할(SAM2) + CVS 를 **끝까지** 학습합니다 — 무료
GPU 로 10시간 넘게 걸리고, 중간에 끊기면 부담입니다. 그 전에 알고 싶은 건
딱 하나:

> **"데이터 → 분할 학습 → CVS 학습 → 벤치마크"가 에러 없이 끝까지 이어지는가?**

이걸 확인하는 게 **스모크 테스트(smoke test)** 입니다. 전자제품을 처음 켤 때
*연기(smoke)가 나는지*만 보는 것에서 온 말 — 성능이 아니라 **"일단 돌아가는가"**
를 봅니다.

## 이 노트북의 선택

- **두 모델 다** 확인합니다: 먼저 가벼운 **U-Net 베이스라인**(02)으로 흐름을
  보고, 그다음 **주력 모델 SAM2 + LoRA**(04)로 *실제 연구 경로*까지 점검합니다.
- **`limit_batches=20`** — 한 epoch 전체가 아니라 20 배치만 돌려, 각 학습이
  **몇 분** 만에 끝납니다. (`src/train/*.py` 가 이 옵션을 trainer 에 넘김)
- 학습이 끝나면 **결과를 눈(예측 그림)과 숫자(벤치마크 표)로 확인**합니다.

⚠️ 적은 배치로 도니 점수는 **당연히 낮습니다.** 여기서 보는 건 점수가 아니라
*"파이프라인이 끝까지 돌았다"* 입니다.

## 순서

1. 환경 + 데이터 준비
2. 분할 학습 (U-Net · 1 epoch · 20 batch)  → 결과 그림 확인
3. CVS 학습 (1 epoch · 위 분할 체크포인트 위에)
4. **(핵심) SAM2 분할도 한 번** → 실제 연구 모델 경로 검증
5. 벤치마크 → U-Net vs SAM2 표 확인
6. 마무리

⚠️ **GPU 런타임 필요** — Runtime → Change runtime type → T4 GPU.
디스크 ~12GB (CholecSeg8k 3.1GB + Endoscapes 5.9GB). Drive 는 쓰지 않고
전부 `/content` 에서 돌립니다 (한 세션에 끝내는 전제).

## 0. 환경 준비

01~05 와 동일 — Colab 은 매번 빈 컴퓨터이므로 코드를 내려받고 라이브러리를
설치합니다. 여러 번 실행해도 안전합니다 (이미 있으면 최신 코드만 당겨옴).

> 코드는 GitHub `main` 에서 받습니다. 아직 머지 안 된 기능 브랜치를 쓰려면
> 아래 `-b main` 을 그 브랜치 이름으로 바꾸세요. (`limit_batches` 옵션이
> 보이지 않으면 코드가 옛 버전이니 이 부분을 확인하세요.)

In [ ]:
%cd /content
import os
if not os.path.isdir("surgical-ai"):
    !git clone -b main https://github.com/duck-bin/surgical-ai.git
%cd surgical-ai
!git pull --ff-only
!bash scripts/colab_setup.sh

import torch
print("준비 완료 ·", os.getcwd())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "없음 → Runtime > Change runtime type > GPU 로 바꾸세요")

## 1. 데이터 준비

batch 를 줄여도 **데이터는 진짜로** 필요합니다 (파이프라인이 데이터를 읽는
부분까지 확인해야 하니까). 두 데이터셋을 `/content` 로 받습니다 — Drive 안 씀.

- **CholecSeg8k** (~3.1GB) → 분할 학습용. HuggingFace 캐시(`./data/cholecseg8k`).
- **Endoscapes2023** (~5.9GB) → CVS 학습용. `/content/endoscapes2023`.

⚠️ 이 셀이 이 노트북에서 **가장 오래 걸립니다** (다운로드 ~20–40분, 네트워크
속도에 따라). 학습 자체보다 다운로드가 길 수 있어요. 한 번 받으면 같은
세션에선 캐시/`-c` 덕분에 다시 안 받습니다.

In [ ]:
# CholecSeg8k (~3.1 GB) -> 로컬 HuggingFace 캐시
!bash scripts/download_cholecseg8k.sh

# Endoscapes2023 (~5.9 GB) -> /content
!wget -c -O /content/endoscapes.zip https://s3.unistra.fr/camma_public/datasets/endoscapes/endoscapes.zip
!unzip -q -o /content/endoscapes.zip -d /content/endoscapes2023

### 데이터가 잘 읽히는지 먼저 확인

학습을 돌리기 전에, 두 로더가 프레임을 제대로 세는지 봅니다. **0 이 아닌
숫자**가 나오면 (CholecSeg8k 합계 8080) 데이터 계층은 정상입니다.

In [ ]:
from src.data.cholecseg8k import CholecSeg8kDataset
from src.data.endoscapes import Endoscapes2023Dataset

for split in ("train", "val", "test"):
    print(f"CholecSeg8k {split:5s}: {len(CholecSeg8kDataset(split=split))} frames")
for split in ("train", "val", "test"):
    ds = Endoscapes2023Dataset(root="/content/endoscapes2023", split=split)
    print(f"Endoscapes  {split:5s}: {len(ds)} CVS keyframes")

## 2. 분할 학습 — U-Net · 1 epoch · 20 batch

이제 **진짜 학습 진입점** `src/train/train_segmentation.py` 를 돌립니다.
02 의 손수 만든 미니 루프와 달리, 이건 검증·체크포인트 저장·스케줄러까지
포함한 *실제* 학습 코드(PyTorch Lightning)입니다.

스모크 테스트용 override 들:

| override | 의미 |
|---|---|
| `model=unet_baseline` | SAM2 대신 가벼운 U-Net 사용 |
| `epochs=1` | 한 epoch 만 |
| `limit_batches=20` | 그 epoch 에서도 **20 배치만** (train/val/test 공통) → 몇 분 |
| `batch_size=4` + `low_memory=false` | per-device 배치 4 · 누적 1 → 20 배치 = **20번 가중치 갱신** |
| `sampler=null loss.class_weighting=none` | 학습 전 **클래스 통계 패스(전체 train 3회 훑기)를 건너뜀** → 스모크 대폭 단축 |
| `data.image_size=256` | 기본 1024 → 256 으로 줄여 로딩·학습 단축 |
| `num_workers=2` | Colab 무료는 vCPU 2개뿐 — 기본 4면 DataLoader 가 느려지거나 멈춤 경고 |

> 💡 **batch 고려**: `batch_size` 는 *효과적 배치*이고, `low_memory` 가 이를
> (per-device 배치 × 누적 스텝)으로 쪼갭니다 (`_batch_and_accumulation`).
> 기본값(batch 16 · low_memory)은 per-device 1 · 누적 16 이라, `limit_batches=20`
> 으로는 **optimizer 스텝이 1번뿐** → 모델이 거의 안 변합니다. 그래서 스모크에선
> 누적을 1 에 가깝게 둬, 20 배치 안에서 갱신이 여러 번 일어나게 합니다.

> ⏱️ **통계 패스 건너뛰기**: 진짜 학습은 시작 전에 클래스 빈도·가중 샘플러를
> 위해 전체 train 을 여러 번 훑습니다(느림, 진행바 없음). 스모크는 품질이
> 목적이 아니므로 `sampler=null loss.class_weighting=none` 으로 이를 끕니다.
> (진짜 학습 `run_pipeline.ipynb` 는 켠 채로 둡니다.)

끝나면 `outputs/unet_baseline/best.ckpt` (와 `last.ckpt`) 가 생기고,
마지막에 `trainer.test` 가 **test_miou** 를 출력합니다 — 그 숫자가 셀
출력 맨 아래에 보이면 분할 단계가 끝까지 돈 것입니다.

⏱️ 통계 패스를 끄면 보통 ~1–2분 (T4 기준).

> ⚠️ **재실행 주의**: 이미 한 번 끝까지 돈 적이 있으면 `outputs/unet_baseline/last.ckpt`
> 가 남아, 다시 실행하면 거기서 **이어받아 곧장 `max_epochs=1 reached` 로 학습을
> 건너뛰고 test 만** 합니다(진행바 안 뜸 — 정상, 이미 학습됨). 처음부터 다시
> 학습하려면 먼저 `!rm -rf outputs/unet_baseline` 로 체크포인트를 지우세요.

> 키워드: effective batch size, gradient accumulation, limit_train_batches

In [ ]:
!python -u -m src.train.train_segmentation \
    model=unet_baseline epochs=1 limit_batches=20 \
    batch_size=4 low_memory=false data.image_size=256 \
    num_workers=2 sampler=null loss.class_weighting=none

## 3. 분할 결과 눈으로 확인

방금 학습한 체크포인트를 불러와, val 프레임 몇 장에 대해 예측을 그려
정답(ground truth)과 비교합니다. **20배치만 학습해 거칠지만**, 예측이 완전
무작위가 아니라 정답의 *형태를 조금이라도 따라가면* 분할 학습이 제대로
연결된 것입니다.

(`load_segmentation_checkpoint` 는 Lightning 체크포인트에서 모델 가중치만
꺼내 주는 헬퍼입니다 — 벤치마크 코드가 쓰는 것과 같은 함수.)

In [ ]:
import torch
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

from src.train.train_segmentation import build_model
from src.eval.benchmark_runner import load_segmentation_checkpoint
from src.data.cholecseg8k import CholecSeg8kDataset, CLASS_NAMES
from src.data.transforms import build_eval_transforms

IMG = 256
device = "cuda" if torch.cuda.is_available() else "cpu"

model_cfg = OmegaConf.load("configs/model/unet_baseline.yaml")
model = build_model(model_cfg, pretrained=False)
model = load_segmentation_checkpoint(model, "outputs/unet_baseline/best.ckpt")
model = model.to(device).eval()

ds = CholecSeg8kDataset(split="val", image_size=IMG,
                        transform=build_eval_transforms(IMG))

N = 3
fig, ax = plt.subplots(N, 3, figsize=(12, 4 * N))
for i in range(N):
    sample = ds[i]
    image = sample["image"].unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(image).argmax(dim=1)[0].cpu()

    rgb = sample["image"].permute(1, 2, 0).numpy()
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
    ax[i, 0].imshow(rgb);                                        ax[i, 0].set_title("input")
    ax[i, 1].imshow(sample["mask"], vmin=0, vmax=5, cmap="tab10"); ax[i, 1].set_title("ground truth")
    ax[i, 2].imshow(pred, vmin=0, vmax=5, cmap="tab10");          ax[i, 2].set_title("U-Net (smoke)")
    for a in ax[i]:
        a.axis("off")
plt.tight_layout()
plt.show()
print("6-class:", CLASS_NAMES)

## 4. CVS 학습 — 위 분할 모델 위에 · 1 epoch · 20 batch

이제 두 번째 단계. `src/train/train_cvs.py` 는 **방금 학습한 분할 모델을
freeze 해서** 6채널 마스크를 만들고, 거기에 RGB 3채널을 더한 9채널을
ViT 분류기에 먹여 CVS 3기준을 학습합니다 (05 에서 본 그 구조).

기본 설정은 분할 모델로 SAM2 를 기대하므로, **우리 U-Net 체크포인트를
가리키도록** override 합니다:

| override | 의미 |
|---|---|
| `data.root=/content/endoscapes2023` | 받아둔 Endoscapes 위치 |
| `segmentation.model_config=configs/model/unet_baseline.yaml` | freeze 할 분할 모델 종류 = U-Net |
| `segmentation.checkpoint=outputs/unet_baseline/best.ckpt` | 2번에서 만든 체크포인트 |
| `batch_size=4` + `low_memory=false` | per-device 배치 4 · 누적 1 → 20번 갱신 (ViT@224 는 T4 에 충분) |
| `epochs=1 limit_batches=20 data.image_size=256` | 빠른 연결 확인 |

끝나면 `outputs/cvs_classifier/best.ckpt` 가 생기고, `trainer.test` 가
**test_map** (mAP) 와 **test_qwk** 를 출력합니다. 그 숫자가 보이면 CVS
단계까지 연결된 것입니다 (값 자체는 낮음).

⏱️ 보통 ~수 분 (T4 기준).

In [ ]:
!python -u -m src.train.train_cvs \
    epochs=1 limit_batches=20 batch_size=4 low_memory=false num_workers=2 \
    data.root=/content/endoscapes2023 data.image_size=256 \
    segmentation.model_config=configs/model/unet_baseline.yaml \
    segmentation.checkpoint=outputs/unet_baseline/best.ckpt

## 5. (핵심) 주력 모델 SAM2 로도 한 번

지금까지는 가벼운 U-Net 으로 *흐름*을 봤습니다. 하지만 실제 연구에 쓰는
모델은 **SAM2 + LoRA** (04). **U-Net 스모크가 통과해도 SAM2 는 전혀 다른
코드 경로**라 따로 확인해야 합니다:

- **LoRA 어댑터** 주입 (PEFT) — U-Net 엔 없는 단계
- **SAM2 base 체크포인트** 다운로드 (`facebook/sam2-hiera-base-plus`)
- **1024 해상도** + `low_memory=true` (per-device 배치 1 · 16x 누적) — T4 메모리 경로
- bf16/fp16 정밀도 폴백

이걸 한 번 돌려봐야 *"진짜 연구 학습이 끝까지 연결되는지"* 확신할 수 있습니다.
역시 `limit_batches=20` + `epochs=1` 로 짧게 끝냅니다. (입력은 256 으로 줘도
SAM2 가 내부에서 1024 로 키워 처리합니다 — `sam2_lora.py` 의 `forward`.)

| override | 의미 |
|---|---|
| `model=sam2_lora` | 주력 모델 (LoRA + SAM2) |
| `batch_size=2` + `low_memory=true` | per-device 배치 **1**(SAM2@1024 는 1장만 올라감) · 누적 2 → 20 배치 = 10번 갱신 |
| `sampler=null loss.class_weighting=none` | 통계 패스 건너뜀 (2번과 동일) |
| `epochs=1 limit_batches=20 data.image_size=256` | 빠른 연결 확인 |

> 💡 SAM2 는 1024 라 per-device 배치는 **1 로 고정**(메모리). `batch_size` 를
> 작게 둬도 forward 횟수(=시간)는 `limit_batches` 가 정하므로 동일하고, **누적만
> 줄어 optimizer 스텝이 늘어납니다** — 그래서 작게 잡았습니다.

⏱️ base 체크포인트 다운로드(첫 1회) 포함 ~3–5분. 끝나면
`outputs/sam2_lora/best.ckpt` 가 생깁니다.

> CVS 를 SAM2 위에서도 돌리려면 4번 셀에서 `segmentation.*` override 를
> `unet_baseline` → `sam2_lora` / `outputs/sam2_lora/best.ckpt` 로 바꾸면 됩니다.

> 키워드: SAM2, LoRA, PEFT, low-memory training, gradient accumulation

In [ ]:
!python -u -m src.train.train_segmentation \
    model=sam2_lora epochs=1 limit_batches=20 \
    batch_size=2 low_memory=true data.image_size=256 \
    num_workers=2 sampler=null loss.class_weighting=none

## 5.5 (선택) SAM2 + temporal — 시간 모델 연결 확인

5번에서 프레임을 **1장씩** 보는 SAM2 + LoRA 를 확인했습니다. 이번엔 그 위에
얹은 **시간(temporal) 모델**(`model=sam2_temporal`)을 점검합니다 — 연속한
**T 프레임 윈도우**를 ConvGRU 로 묶어 *타깃(마지막) 프레임*의 예측을 다듬는
변형입니다 (목표: cystic_duct recall + 프레임 간 일관성).

**왜 또 따로 보나** — SAM2 스모크가 통과해도 temporal 은 *추가 코드 경로*라
따로 확인해야 합니다:

- **윈도우 데이터셋** `CholecSeg8kWindowDataset` — 같은 비디오 안에서만 연속
  프레임 윈도우를 만듦 (비디오·train/val/test 경계를 절대 안 넘음)
- **clip 입력** `(B, T, 3, H, W)` → 출력은 그대로 `(B, C, H, W)` (타깃 프레임)
  이라 `SegmentationModule` 에 그대로 꽂힘
- **ConvGRU 시간 헤드** + 자체 LR 그룹 (인코더 LoRA 는 5번과 동일하게 유지)

| override | 의미 |
|---|---|
| `model=sam2_temporal` | 시간 모델 (SAM2 + LoRA + ConvGRU) |
| `model.window=2` | **작은 윈도우**(연속 2프레임) — 스모크라 짧게 |
| `batch_size=2` + `low_memory=true` | per-device 1 · 누적 2 (윈도우당 SAM2 forward 가 T배라 가볍게) |
| `sampler=null loss.class_weighting=none` | 통계 패스 건너뜀 (5번과 동일; temporal 은 가중 샘플러 미사용) |
| `epochs=1 limit_batches=20 data.image_size=256` | 빠른 연결 확인 |

끝나면 `outputs/sam2_temporal/best.ckpt` 가 생기고 `trainer.test` 가
**test_miou** 를 출력합니다. `configs/benchmark.yaml` 에 이미 한 행이 있어,
6번 벤치마크가 이 체크포인트를 **자동으로 한 행 더** 채웁니다 — 이 셀을
건너뛰면 그 행은 `TBD` 로 남습니다 (벤치마크는 빠진 체크포인트를 건너뜀).

⏱️ 윈도우당 SAM2 를 T번 도므로 5번보다 살짝 더 걸립니다 (~수 분, T4).
입력 256 은 SAM2 가 내부에서 1024 로 키워 처리합니다 (5번과 동일).

> 키워드: temporal modeling, ConvGRU, sliding window, video-level split

In [ ]:
!python -u -m src.train.train_segmentation \
    model=sam2_temporal epochs=1 limit_batches=20 \
    model.window=2 batch_size=2 low_memory=true data.image_size=256 \
    num_workers=2 sampler=null loss.class_weighting=none

## 6. 벤치마크 — U-Net vs SAM2 비교표

이제 U-Net 과 SAM2 체크포인트가 **둘 다** 있으니, 벤치마크가 두 행을 모두
채웁니다. `src/eval/benchmark_runner.py` 가 CholecSeg8k 테스트셋에서 평가해
비교표(mIoU, Cystic Duct Dice, 부트스트랩 95% 신뢰구간)를
`results/benchmark_table.md` 에 씁니다.

스모크용이라 `max_eval_batches=40` 으로 **앞 40프레임만** 빠르게 평가합니다
(점수는 참고용 — 표본이 적어 노이즈가 큽니다). SAM2 는 1024 추론이라
`batch_size=1` 로 둡니다.

> 진짜 비교(전체 테스트셋 + 부트스트랩 CI)는 충분히 학습한 뒤
> `run_pipeline.ipynb` 의 벤치마크로 얻습니다.

In [ ]:
!python -u -m src.eval.benchmark_runner data.image_size=256 batch_size=1 max_eval_batches=40

from pathlib import Path
from IPython.display import Markdown, display

table = Path("results/benchmark_table.md").read_text()
print(table)
display(Markdown(table))

## 마무리 — 이 노트북 정리

### 무엇을 확인했나
이 노트북이 에러 없이 끝까지 돌았다면, **파이프라인 전체가 (두 모델 모두)
연결**된 것입니다:

```
데이터 다운로드  ✅
   → 분할 학습 U-Net  (1 epoch · 20 batch)   → outputs/unet_baseline/best.ckpt  ✅
   → CVS 학습         (위 체크포인트 freeze)  → outputs/cvs_classifier/best.ckpt ✅
   → 분할 학습 SAM2   (LoRA · 1024 · low_mem) → outputs/sam2_lora/best.ckpt      ✅
   → 벤치마크                                 → results/benchmark_table.md       ✅
```

### 기억할 점
- **스모크 테스트 = "돌아가는가"** 확인. 적은 배치의 낮은 점수는 *정상*이며
  목표가 아니다. (좋은 점수는 충분한 학습에서 나온다)
- 진짜 학습 코드를 **override** 로 조절했다: `model=`, `epochs=`,
  `limit_batches=`, `batch_size=`, `low_memory=`, `data.image_size=`,
  `segmentation.checkpoint=` 등. `limit_batches` 는 이번에 추가한 옵션으로,
  한 epoch 의 배치 수를 잘라 *어떤 모델이든* 몇 분 만에 점검하게 해 준다.
- **batch 도 고려**: `batch_size`+`low_memory` 가 (per-device 배치 × 누적)을
  정한다. 누적이 크면 `limit_batches` 안의 optimizer 갱신이 줄어드니, 스모크에선
  누적을 작게 잡아 갱신이 실제로 여러 번 일어나게 했다.
- **U-Net 통과 ≠ SAM2 통과**: SAM2 는 LoRA·1024·low-memory 등 다른 경로라
  따로 확인해야 진짜 연구 학습을 신뢰할 수 있다.
- CVS 단계는 **앞 단계의 분할 체크포인트**에 의존한다.

### 다음 단계 — 자신이 생겼다면
- **제대로 학습**: `limit_batches` 를 빼고 `epochs` 를 키워서 (예: SAM2
  `epochs=100`) 점수가 올라가는지. 이건 `run_pipeline.ipynb` 가 끝까지 합니다.
- 벤치마크 표에서 **U-Net(베이스라인) 대비 SAM2 의 이득**을 본다 (02 의 주제).
- 데모: `app/gradio_demo.py` — 프레임을 올리면 분할 + CVS 점수를 보여준다.